# Final Project · Dự đoán kết quả trận League of Legends

Đây là **bài nộp chấm điểm** của khóa. Notebook này là **khung để bạn tự làm**: đề từng bước (viết bằng lời của Claude, bám đúng 8 nhiệm vụ của khóa), ô code để bạn viết, 3 tầng gợi ý, và ô **tự kiểm tra hình dạng/kiểu dữ liệu** (chỉ kiểm tra quy trình, không kiểm tra con số kết quả).

**Không có lời giải trong repo.** Bài chấm điểm phải là sản phẩm của chính bạn. Kẹt thì mở **Gợi ý 1**, rồi **Gợi ý 2**, cuối cùng mới **Gợi ý 3** (gần như lời giải, vẫn thiếu vài chỗ). Vẫn kẹt thì hỏi Claude, bạn sẽ được gợi ý thêm.

Đọc [`brief.md`](brief.md) trước: có thông tin quan trọng về dữ liệu (đừng mong độ chính xác cao) và về cách nộp.

Môi trường do **uv** quản lý: trong thư mục khóa `04-intro-neural-networks-pytorch/`, chạy `uv sync` (một lần), rồi `uv run jupyter lab`.

## Chuẩn bị

In [ ]:
import itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc as auc_score
%matplotlib inline

torch.manual_seed(0)
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/rk7VDaPjMp1h5VXS-cUyMg/league-of-legends-data-large.csv"
print("PyTorch", torch.__version__)

## Bài 1. Bước 1 · Tải và tiền xử lý dữ liệu

1. Đọc CSV từ `URL` vào DataFrame `data`.
2. Tách `X` (mọi cột trừ `win`) và `y` (cột `win`: 1 = thắng, 0 = thua).
3. Chia 80/20 bằng `train_test_split` với `random_state=42`.
4. Chuẩn hóa đặc trưng bằng `StandardScaler`, **chỉ học thống kê từ tập train**.
5. Đổi sang tensor `float32`: `X_train_t`, `X_test_t`; nhãn `y_train_t`, `y_test_t` có shape `(N, 1)` (vì `BCELoss` cần cùng shape với đầu ra của mô hình).

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Quy trình gồm đúng những bước bạn đã làm ở Practice Project. Chỉ có hai điểm cần cẩn thận: (a) scaler học từ đâu, (b) shape của nhãn.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`pd.read_csv`, `data.drop('win', axis=1)`, `train_test_split(..., test_size=0.2, random_state=42)`, `scaler.fit_transform` rồi `scaler.transform`. Nhãn: `torch.tensor(y.values, dtype=torch.float32).view(-1, 1)`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`X_train_t = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32)`; `X_test_t = torch.tensor(scaler.____(X_test), dtype=torch.float32)`.

</details>

In [ ]:
# Viết code của bạn ở đây.
# Cần tạo ra các biến: data, X, y, X_train, X_test, y_train, y_test, scaler,
#                      X_train_t, X_test_t, y_train_t, y_test_t

In [ ]:
assert tuple(X_train_t.shape) == (800, 8) and tuple(X_test_t.shape) == (200, 8)
assert tuple(y_train_t.shape) == (800, 1) and tuple(y_test_t.shape) == (200, 1)
assert X_train_t.dtype == torch.float32 and y_train_t.dtype == torch.float32
assert torch.allclose(X_train_t.mean(dim=0), torch.zeros(8), atol=1e-4), "train phải có trung bình 0 sau chuẩn hóa"
assert set(y_train_t.unique().tolist()) <= {0.0, 1.0}
print("✅ Bước 1: hình dạng và kiểu dữ liệu hợp lệ")

## Bài 2. Bước 2 · Mô hình logistic regression

1. Viết lớp `LogisticRegressionModel(nn.Module)`: `__init__(self, input_dim)` tạo `self.linear = nn.Linear(input_dim, 1)`; `forward` trả sigmoid của đầu ra lớp tuyến tính.
2. `input_dim = X_train_t.shape[1]`; tạo `model`.
3. Hàm loss `criterion = nn.BCELoss()`; optimizer `optimizer = optim.SGD(model.parameters(), lr=0.01)`.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Bạn đã viết đúng lớp này ở Module 6, Bài 1. Hãy nhớ lại `forward` có thêm bước nào so với hồi quy tuyến tính.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`torch.sigmoid(self.linear(x))`. Số đặc trưng lấy từ shape của dữ liệu thay vì gõ cứng.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`input_dim = X_train_t.shape[____]`; `optimizer = optim.SGD(model.parameters(), lr=____)`.

</details>

In [ ]:
# Viết code của bạn ở đây.
# Cần tạo: lớp LogisticRegressionModel, input_dim, model, criterion, optimizer

In [ ]:
with torch.no_grad():
    out = model(X_train_t)
assert tuple(out.shape) == (800, 1) and ((out > 0) & (out < 1)).all(), "đầu ra phải là xác suất trong (0, 1)"
assert tuple(model.linear.weight.shape) == (1, 8)
assert isinstance(criterion, nn.BCELoss)
assert optimizer.param_groups[0]["lr"] == 0.01
print("✅ Bước 2: mô hình, loss và optimizer hợp lệ")

## Bài 3. Bước 3 · Huấn luyện mô hình

Huấn luyện **1000 epoch** (mỗi epoch dùng toàn bộ tập train một lần). Mỗi epoch: `model.train()`, `optimizer.zero_grad()`, dự đoán, tính loss, `backward()`, `optimizer.step()`. Lưu loss mỗi epoch vào list `losses` và **in loss mỗi 100 epoch**.
Sau đó: `model.eval()`, dưới `torch.no_grad()` dự đoán trên train và test, tính **độ chính xác** (ngưỡng 0.5) vào `train_acc`, `test_acc`, và in ra.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Khung huấn luyện giống Module 3 và Module 6. Chỉ cần nhớ ba thứ: xóa gradient, tính loss, cập nhật; và đánh giá thì không cập nhật.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`if (epoch + 1) % 100 == 0: print(...)`. Độ chính xác: `((p >= 0.5).float() == y).float().mean().item()`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

Với `model.eval()` và `torch.no_grad()`: `pred_test = model(X_test_t)`; `test_acc = ((pred_test >= 0.5).float() == y_test_t).float().mean().item()`.

</details>

In [ ]:
# Viết code của bạn ở đây.
# Cần tạo: losses (list 1000 phần tử), train_acc, test_acc

In [ ]:
assert len(losses) == 1000
assert losses[-1] < losses[0], "loss phải giảm qua huấn luyện"
assert 0.0 <= train_acc <= 1.0 and 0.0 <= test_acc <= 1.0
print("✅ Bước 3: quy trình huấn luyện hợp lệ (đừng mong độ chính xác cao: xem brief.md)")

## Bài 4. Bước 4 · Tối ưu: L2 regularization

Tạo mô hình **mới** `model_l2` và optimizer `optim.SGD` có **`weight_decay=0.01`** (L2 regularization), cùng `lr=0.01`. Huấn luyện lại 1000 epoch như bước 3 rồi lưu `train_acc_l2`, `test_acc_l2`.
So sánh với bước 3: điều gì thay đổi (và điều gì không)?

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Regularization không đòi thêm lớp hay thêm dữ liệu. Nó can thiệp vào cách optimizer cập nhật trọng số.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

Tham số `weight_decay` của `optim.SGD`. Nhớ khởi tạo mô hình **mới** để optimizer mới gắn với tham số mới.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`optimizer_l2 = optim.SGD(model_l2.parameters(), lr=0.01, weight_decay=____)`.

</details>

In [ ]:
# Viết code của bạn ở đây.
# Cần tạo: model_l2, optimizer_l2, train_acc_l2, test_acc_l2

In [ ]:
assert optimizer_l2.param_groups[0]["weight_decay"] == 0.01
assert 0.0 <= train_acc_l2 <= 1.0 and 0.0 <= test_acc_l2 <= 1.0
print("✅ Bước 4: optimizer có L2 regularization")

## Bài 5. Bước 5 · Trực quan hóa: confusion matrix và ROC

Với mô hình đã huấn luyện (dùng `model_l2` hoặc `model`, tùy bạn, hãy ghi rõ bạn chọn cái nào):
1. `cm`: **confusion matrix** trên tập test (`sklearn.metrics.confusion_matrix`), vẽ bằng `plt.imshow` kèm số trong từng ô.
2. In **classification report**.
3. `roc_auc`: **diện tích dưới đường ROC** (`roc_curve` rồi `auc`), và vẽ đường ROC cùng đường chéo tham chiếu.
Rồi trả lời bằng chữ trong một ô markdown: AUC gần bao nhiêu và điều đó nói gì về khả năng dự đoán của mô hình?

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Confusion matrix đếm bốn loại kết quả (đúng/sai × dương/âm). ROC đo sự đánh đổi giữa bắt đúng và báo nhầm khi thay đổi ngưỡng.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`y_pred = (model(X_test_t) >= 0.5).float()`. Xác suất (không qua ngưỡng) mới dùng được cho ROC: `y_prob`. Nhớ chuyển sang NumPy: `.numpy().ravel()`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`fpr, tpr, _ = roc_curve(y_test_t.numpy().ravel(), y_prob)`; `roc_auc = auc_score(fpr, tpr)`.

</details>

In [ ]:
# Viết code của bạn ở đây.
# Cần tạo: cm (ma trận 2x2), roc_auc (số), và hai hình vẽ

In [ ]:
assert cm.shape == (2, 2) and cm.sum() == len(y_test_t) == 200
assert 0.0 <= roc_auc <= 1.0
print(f"✅ Bước 5: hợp lệ (AUC của bạn: {roc_auc:.3f})")

## Bài 6. Bước 6 · Lưu và nạp mô hình

1. Chọn mô hình tốt nhất của bạn làm `final_model`.
2. Lưu `state_dict` vào file `lol_logreg.pth` (`torch.save`).
3. Tạo `loaded_model = LogisticRegressionModel(input_dim)`, nạp tham số (`load_state_dict(torch.load(...))`), `eval()`.
4. Đánh giá `loaded_model` trên tập test (độ chính xác `loaded_acc`) và xác nhận nó cho **cùng dự đoán** như `final_model`.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Chỉ lưu tham số, không lưu cả lớp. Mô hình mới phải cùng kiến trúc để nhận được tham số.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`torch.save(final_model.state_dict(), 'lol_logreg.pth')`; sau khi nạp nhớ `loaded_model.eval()`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`loaded_model.load_state_dict(torch.load('lol_logreg.pth'))`.

</details>

In [ ]:
# Viết code của bạn ở đây.
# Cần tạo: final_model, loaded_model, loaded_acc

In [ ]:
import os
assert os.path.exists("lol_logreg.pth")
final_model.eval()
with torch.no_grad():
    assert torch.allclose(loaded_model(X_test_t), final_model(X_test_t)), "mô hình nạp lại phải cho cùng dự đoán"
assert 0.0 <= loaded_acc <= 1.0
print("✅ Bước 6: lưu và nạp thành công")

## Bài 7. Bước 7 · Chọn learning rate

Thử `learning_rates = [0.01, 0.05, 0.1]`. Với mỗi giá trị: **khởi tạo lại mô hình và optimizer** (`optim.SGD`), huấn luyện một số epoch cố định (ví dụ 100), rồi ghi độ chính xác test.
Lưu `results` (dict: learning rate → độ chính xác test) và `best_lr` (learning rate có độ chính xác test cao nhất).
Nhận xét: nếu các độ chính xác gần như bằng nhau thì kết luận "learning rate nào tốt nhất" đáng tin đến đâu?

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Mỗi learning rate cần một mô hình mới, nếu không bạn đang so sánh các mô hình đã được huấn luyện với lịch sử khác nhau.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

Vòng `for lr in learning_rates:` bao ngoài vòng huấn luyện. Chọn lớn nhất: `max(results, key=results.get)`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

Trong vòng ngoài: `m = LogisticRegressionModel(input_dim)`, `opt = optim.SGD(m.parameters(), lr=lr)`.

</details>

In [ ]:
# Viết code của bạn ở đây.
# Cần tạo: learning_rates, results (dict), best_lr

In [ ]:
assert set(results) == {0.01, 0.05, 0.1}
assert all(0.0 <= v <= 1.0 for v in results.values())
assert best_lr in results and results[best_lr] == max(results.values())
print("✅ Bước 7: hợp lệ")

## Bài 8. Bước 8 · Mức độ quan trọng của đặc trưng

Lấy trọng số của lớp tuyến tính trong mô hình tốt nhất của bạn.
1. `importance_df`: DataFrame hai cột `Feature` (tên đặc trưng, lấy từ `X_train.columns`) và `Importance` (trọng số tương ứng).
2. Sắp xếp theo **giá trị tuyệt đối** của `Importance` giảm dần.
3. Vẽ biểu đồ cột ngang.
4. Viết vài dòng diễn giải: trọng số dương/âm nghĩa là gì; vì sao khi tất cả đặc trưng đã **chuẩn hóa** thì so sánh độ lớn trọng số mới có ý nghĩa?

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Trọng số của mô hình tuyến tính cho biết mỗi đặc trưng đẩy dự đoán về phía nào và mạnh đến đâu, với điều kiện các đặc trưng cùng thang đo.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`final_model.linear.weight.detach().numpy().ravel()`. Sắp xếp: `df.reindex(df['Importance'].abs().sort_values(ascending=False).index)`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`importance_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': weights})`.

</details>

In [ ]:
# Viết code của bạn ở đây.
# Cần tạo: importance_df (đã sắp xếp theo |Importance| giảm dần) và biểu đồ

In [ ]:
assert list(importance_df.columns) == ["Feature", "Importance"]
assert len(importance_df) == 8 and set(importance_df["Feature"]) == set(X_train.columns)
imp = importance_df["Importance"].abs().tolist()
assert imp == sorted(imp, reverse=True), "phải sắp xếp theo |Importance| giảm dần"
print("✅ Bước 8: hợp lệ")

## Xong rồi
Đối chiếu với `solution/`, rồi quay lại `lesson.md` nếu còn chỗ chưa chắc.